# 统一清洗：发票 CSV → 回归数据

**输入文件**（均位于 `G:\Kuangyu_Temp\Outsource\productivity\`）：

| 文件 | 内容 | 颗粒度 | 口径 |
|---|---|---|---|
| `firm_buy.csv` | 样本企业采购 | 购方企业ID × 项目代码 | 样本 |
| `firm_sell.csv` | 样本企业销售 | 销方企业ID × 项目代码 | 样本 |
| `firm_city.csv` | 样本企业地区对照 | 企业ID × 地区 | 样本 |
| `city_buy.csv` | 全城市采购 | 购方地区 × 项目代码 | **全量** |
| `city_sell.csv` | 全城市销售 | 销方地区 × 项目代码 | **全量** |
| `bianma.dta` | 合法 9 位产品码（2778 个） | — | — |
| `full_data.dta` | 企业特征 | firm × year | — |

**输出文件**（落在同目录）：

| 文件 | 内容 |
|---|---|
| `invoice_panel.dta` | 主回归面板：firm × product × city × year，DV = `ln_p_net` |
| `market_conds.dta` | 市场条件：product × city × year（买方/卖方均来自全量） |
| `firm_chars.dta` | 企业特征：firm × year |

**关键设计**：
- `firm_buy.csv` 和 `firm_sell.csv` 本身不带地区，需用 `firm_city.csv` 合并得到 `city`
- 采购价格面板中的 `city` 来自样本企业所在地区
- `city_buy.csv` 使用 `购方地区`，`city_sell.csv` 使用 `销方地区`
- 市场条件（`n_buyers`, `n_sellers`, `p_mkt` 等）来自全量城市聚合数据

In [30]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

BASE = Path(r'G:\Kuangyu_Temp\Outsource\productivity')
ROOT = BASE.parent

print('BASE:', BASE)
print('ROOT:', ROOT)

BASE: G:\Kuangyu_Temp\Outsource\productivity
ROOT: G:\Kuangyu_Temp\Outsource


## Step 1　读入所有 CSV

ID、地区代码、项目代码全部按字符串读入，避免长数字被截断。

In [31]:
NA = ['', 'NULL', '(Null)', 'null', 'None', 'nan', 'NaN', '#N/A']

firm_buy  = pd.read_csv(BASE / 'firm_buy.csv',  dtype=str, na_values=NA, encoding='utf-8-sig')
firm_sell = pd.read_csv(BASE / 'firm_sell.csv', dtype=str, na_values=NA, encoding='utf-8-sig')
firm_city = pd.read_csv(BASE / 'firm_city.csv', dtype=str, na_values=NA, encoding='utf-8-sig')
city_buy  = pd.read_csv(BASE / 'city_buy.csv',  dtype=str, na_values=NA, encoding='utf-8-sig')
city_sell = pd.read_csv(BASE / 'city_sell.csv', dtype=str, na_values=NA, encoding='utf-8-sig')

for name, df in [('firm_buy', firm_buy), ('firm_sell', firm_sell), ('firm_city', firm_city),
                 ('city_buy', city_buy), ('city_sell', city_sell)]:
    df.columns = df.columns.str.strip()
    print(f'{name:12s}  shape={df.shape}  cols={list(df.columns)}')

firm_buy      shape=(473268, 4)  cols=['购方企业ID', '项目代码', '金额合计', '数量合计']
firm_sell     shape=(101844, 4)  cols=['销方企业ID', '项目代码', '金额合计', '数量合计']
firm_city     shape=(3410, 2)  cols=['企业ID', '地区']
city_buy      shape=(2104901, 5)  cols=['购方地区', '项目代码', '买方企业数', '金额合计', '数量合计']
city_sell     shape=(1540065, 5)  cols=['销方地区', '项目代码', '卖方企业数', '金额合计', '数量合计']


## Step 2　统一列名，并合并样本企业地区

`firm_buy` 和 `firm_sell` 本身没有地区列，需要从 `firm_city.csv` 按企业 ID 合并地区。

In [32]:
firm_city_map = firm_city.rename(columns={
    '企业ID': 'firm_id', '地区': 'city'
})[['firm_id', 'city']].copy()
firm_city_map['firm_id'] = firm_city_map['firm_id'].astype('string').str.strip()
firm_city_map['city'] = firm_city_map['city'].astype('string').str.strip()

fb = firm_buy.rename(columns={
    '购方企业ID': 'firm_id',
    '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'
})[['firm_id', 'product_code', 'value', 'qty']].copy()
fb['firm_id'] = fb['firm_id'].astype('string').str.strip()
fb = fb.merge(firm_city_map, on='firm_id', how='left', validate='m:1')
fb = fb[['firm_id', 'city', 'product_code', 'value', 'qty']].copy()

fs = firm_sell.rename(columns={
    '销方企业ID': 'firm_id',
    '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'
})[['firm_id', 'product_code', 'value', 'qty']].copy()
fs['firm_id'] = fs['firm_id'].astype('string').str.strip()
fs = fs.merge(firm_city_map, on='firm_id', how='left', validate='m:1')
fs = fs[['firm_id', 'city', 'product_code', 'value', 'qty']].copy()

# city_buy: 全量采购侧，含买方企业数
cb_cols = {'购方地区': 'city', '项目代码': 'product_code',
           '买方企业数': 'n_buyers_raw', '金额合计': 'value', '数量合计': 'qty'}
cb = city_buy.rename(columns=cb_cols)[list(cb_cols.values())].copy()

# city_sell: 全量供给侧，按销方地区汇总，含卖方企业数
cs_cols = {'销方地区': 'city', '项目代码': 'product_code',
           '卖方企业数': 'n_sellers_raw', '金额合计': 'value', '数量合计': 'qty'}
cs = city_sell.rename(columns=cs_cols)[list(cs_cols.values())].copy()

for name, df in [('fb', fb), ('fs', fs), ('cb', cb), ('cs', cs)]:
    print(f'{name}  {df.shape}')

fb  (473268, 5)
fs  (101844, 5)
cb  (2104901, 5)
cs  (1540065, 5)


## Step 3　转换数值、清洗项目代码

对四个数据集统一执行：
1. value / qty 转数值（`errors='coerce'`）
2. 项目代码必须纯数字
3. 右补零到 19 位，取前 9 位 → `product_id`
4. 删除 value ≤ 0 或 qty ≤ 0 的行（含红冲净值为负的记录）

In [33]:
def clean_codes(df, label):
    n0 = len(df)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['qty']   = pd.to_numeric(df['qty'],   errors='coerce')
    # 纯数字项目代码
    df = df.dropna(subset=['product_code', 'value', 'qty']).copy()
    df = df[df['product_code'].str.fullmatch(r'\d+', na=False)].copy()
    # 右补零 → 19 位 → 取前 9 位
    df['product_id'] = df['product_code'].str.ljust(19, '0').str[:9]
    # 删除非正
    df = df[(df['value'] > 0) & (df['qty'] > 0)].copy()
    print(f'{label:6s}: {n0:>10,} → {len(df):>10,}  unique products: {df["product_id"].nunique()}')
    return df

fb = clean_codes(fb, 'fb')
fs = clean_codes(fs, 'fs')
cb = clean_codes(cb, 'cb')
cs = clean_codes(cs, 'cs')

fb    :    473,268 →    441,318  unique products: 5540
fs    :    101,844 →     94,093  unique products: 3872
cb    :  2,104,901 →  1,920,019  unique products: 332395
cs    :  1,540,065 →  1,386,189  unique products: 332390


## Step 4　匹配 bianma — 只保留合法 9 位产品码

In [34]:
bianma = pd.read_stata(ROOT / 'bianma.dta')
valid = set(bianma['product_id'].astype(str).str.strip())
print(f'bianma: {len(valid)} valid products')

for name, df in [('fb', fb), ('fs', fs), ('cb', cb), ('cs', cs)]:
    n = len(df)
    df = df[df['product_id'].isin(valid)].copy()
    print(f'{name}: {n:>10,} → {len(df):>10,}  match {len(df)/n*100:.1f}%'
          f'  unique products: {df["product_id"].nunique()}')
    if name == 'fb': fb = df
    if name == 'fs': fs = df
    if name == 'cb': cb = df
    if name == 'cs': cs = df

bianma: 2778 valid products
fb:    441,318 →    425,713  match 96.5%  unique products: 2731
fs:     94,093 →     89,584  match 95.2%  unique products: 2542
cb:  1,920,019 →  1,393,329  match 72.6%  unique products: 2778
cs:  1,386,189 →    954,222  match 68.8%  unique products: 2778


## Step 5　构造 DV：企业采购价格面板

`firm_buy` 是企业 × 产品的年度聚合；这里先用 `firm_city.csv` 补上企业所在地区，再按企业 × 地区 × 产品聚合一次以防重复行，同时加 year。

**DV = `p_buy` = 金额合计 / 数量合计**（用`ln_p_net`作为别名，保持与Stata脚本兼容）。

In [35]:
fb['year'] = 2017

dv = fb.groupby(['firm_id', 'product_id', 'city', 'year'], as_index=False).agg(
    value  = ('value', 'sum'),
    qty    = ('qty',   'sum'),
    n_rows = ('value', 'size')
)
dv = dv[(dv['value'] > 0) & (dv['qty'] > 0)].copy()


dv['p_buy']    = dv['value'] / dv['qty']
dv['ln_p_buy'] = np.log(dv['p_buy'])
dv['ln_p_net'] = dv['ln_p_buy']   # 兼容 Stata 脚本
dv['p_net']    = dv['p_buy']
dv['ln_qty']   = np.log(dv['qty'])

print('invoice_panel rows:', len(dv))
print('unique firms:   ', dv['firm_id'].nunique())
print('unique products:', dv['product_id'].nunique())
print('unique cities:  ', dv['city'].nunique())
dv[['p_buy', 'qty', 'value']].describe(percentiles=[.01, .05, .5, .95, .99]).round(2)

invoice_panel rows: 403400
unique firms:    3376
unique products: 2731
unique cities:   319


,p_buy,qty,value
count,4.034000e+05,4.034000e+05,4.034000e+05
mean,2.720558e+04,1.256130e+06,1.094831e+07
std,3.913541e+06,1.443377e+08,4.264354e+08
min,0.000000e+00,0.000000e+00,0.000000e+00
1%,2.000000e-01,1.000000e+00,1.695000e+01
5%,1.560000e+00,1.000000e+00,1.041300e+02
50%,8.673000e+01,1.030000e+02,2.060000e+04
95%,2.518539e+04,3.202603e+05,8.861274e+06
99%,2.133625e+05,6.104243e+06,1.009085e+08
max,2.361583e+09,6.523459e+10,1.089544e+11


## Step 6　市场条件（买方侧）

来自全量 `city_buy.csv`，按 `city × product_id × year` 聚合。

- `mkt_value`, `mkt_qty`, `p_mkt`：全市场采购金额、数量、均价
- `n_buyers`：买方企业数（来自 city_buy 的 `买方企业数`，因已按 19→9 位合并，可能轻微重复计数，参见 city_market_condition_note.md §7.1）

In [36]:
cb['year']          = 2017
cb['n_buyers_raw']  = pd.to_numeric(cb['n_buyers_raw'], errors='coerce')

mkt_buy = cb.groupby(['product_id', 'city', 'year'], as_index=False).agg(
    mkt_value  = ('value',        'sum'),
    mkt_qty    = ('qty',          'sum'),
    n_buyers   = ('n_buyers_raw', 'sum'),
    n_raw_buy  = ('product_code', 'nunique')
)
mkt_buy = mkt_buy[(mkt_buy['mkt_value'] > 0) & (mkt_buy['mkt_qty'] > 0)].copy()
mkt_buy['p_mkt']       = mkt_buy['mkt_value'] / mkt_buy['mkt_qty']
mkt_buy['ln_p_mkt']    = np.log(mkt_buy['p_mkt'])
mkt_buy['ln_mkt_qty']  = np.log(mkt_buy['mkt_qty'])
mkt_buy['ln_n_buyers'] = np.log(mkt_buy['n_buyers'].clip(lower=1))

print('mkt_buy rows:', len(mkt_buy))
print('unique products:', mkt_buy['product_id'].nunique())
print('unique cities:  ', mkt_buy['city'].nunique())
mkt_buy[['mkt_qty', 'p_mkt', 'n_buyers']].describe(percentiles=[.01, .5, .99]).round(2)

mkt_buy rows: 1066579
unique products: 2778
unique cities:   1174


,mkt_qty,p_mkt,n_buyers
count,1.066579e+06,1.066579e+06,1066579.00
mean,3.675879e+10,1.587276e+04,183.92
std,2.253587e+13,8.464837e+05,1320.09
min,0.000000e+00,0.000000e+00,1.00
1%,1.000000e+00,3.100000e-01,1.00
50%,1.265600e+04,8.089000e+01,15.00
99%,2.592709e+08,1.666964e+05,2948.00
max,2.000000e+16,5.983488e+08,184334.00


## Step 7　市场条件（卖方侧）

来自全量 `city_sell.csv`，按**销方地区** × 产品汇总。

经济含义：`n_sellers` = 在城市 c，有多少不同卖方企业销售 product p。

In [37]:
cs['year']           = 2017
cs['n_sellers_raw']  = pd.to_numeric(cs['n_sellers_raw'], errors='coerce')

mkt_sell = cs.groupby(['product_id', 'city', 'year'], as_index=False).agg(
    sell_value  = ('value',          'sum'),
    sell_qty    = ('qty',            'sum'),
    n_sellers   = ('n_sellers_raw',  'sum'),
    n_raw_sell  = ('product_code',   'nunique')
)
mkt_sell = mkt_sell[(mkt_sell['sell_value'] > 0) & (mkt_sell['sell_qty'] > 0)].copy()
mkt_sell['ln_n_sellers'] = np.log(mkt_sell['n_sellers'].clip(lower=1))

print('mkt_sell rows:', len(mkt_sell))
print('unique products:', mkt_sell['product_id'].nunique())
print('unique cities:  ', mkt_sell['city'].nunique())
mkt_sell[['sell_qty', 'n_sellers']].describe(percentiles=[.01, .5, .99]).round(2)

mkt_sell rows: 793365
unique products: 2778
unique cities:   1004


,sell_qty,n_sellers
count,7.933650e+05,793365.00
mean,4.941759e+10,53.70
std,2.612968e+13,282.11
min,0.000000e+00,1.00
1%,1.000000e+00,1.00
50%,9.799190e+03,8.00
99%,3.395308e+08,810.00
max,2.000000e+16,36287.00


## Step 8　合并市场条件

以买方侧为基础（`mkt_buy`），left join 卖方侧（`mkt_sell`）。缺失的 `n_sellers` 在后面诊断时会看到比例。

In [38]:
market_conds = mkt_buy.merge(
    mkt_sell[['product_id', 'city', 'year', 'sell_value', 'sell_qty', 'n_sellers', 'ln_n_sellers']],
    on=['product_id', 'city', 'year'],
    how='left'
)

print('market_conds rows:', len(market_conds))
print('n_sellers missing:', market_conds['n_sellers'].isna().mean())

market_conds rows: 1066579
n_sellers missing: 0.2744456810044075


## Step 9　企业特征（来自 full_data.dta）

分块读取避免 OOM。只取 firm × year 级别的变量：规模、产品数、是否中介。

In [39]:
usecols = ['firm_id', 'year', 'firm_total_output', 'firm_total_outsource',
           'n_products', 'is_intermediary']

parts = []
for ch in pd.read_stata(ROOT / 'full_data.dta', columns=usecols, chunksize=500_000):
    parts.append(ch.drop_duplicates(['firm_id', 'year']))

firm_chars = pd.concat(parts, ignore_index=True).drop_duplicates(['firm_id', 'year'])
firm_chars['firm_id'] = firm_chars['firm_id'].astype(str).str.strip()
firm_chars['year']    = firm_chars['year'].astype(int)
firm_chars['ln_firm_output']    = np.log(firm_chars['firm_total_output'].clip(lower=1))
firm_chars['ln_firm_outsource'] = np.log(firm_chars['firm_total_outsource'].clip(lower=1))

print('firm_chars rows:', len(firm_chars))
print('unique firms:   ', firm_chars['firm_id'].nunique())

firm_chars rows: 12339537
unique firms:    7191877


## Step 10　覆盖率诊断

检查三个 merge 的覆盖率，确认没有大规模样本损失。

In [40]:
# DV × 市场条件
tmp = dv.merge(
    market_conds[['product_id', 'city', 'year', 'ln_p_mkt', 'ln_n_buyers', 'ln_n_sellers']],
    on=['product_id', 'city', 'year'], how='left', indicator=True
)
print('DV × market_conds:')
print(tmp['_merge'].value_counts())
print(f'  market missing: {(tmp["_merge"] != "both").mean()*100:.1f}%')
print(f'  ln_n_sellers missing among matched: {tmp["ln_n_sellers"].isna().mean()*100:.1f}%')

# DV × 企业特征
tmp2 = dv.merge(
    firm_chars[['firm_id', 'year', 'ln_firm_output', 'is_intermediary']],
    on=['firm_id', 'year'], how='left', indicator=True
)
print('\nDV × firm_chars:')
print(tmp2['_merge'].value_counts())
print(f'  firm_chars missing: {(tmp2["_merge"] != "both").mean()*100:.1f}%')

DV × market_conds:
_merge
both          403146
left_only        254
right_only         0
Name: count, dtype: int64
  market missing: 0.1%
  ln_n_sellers missing among matched: 1.3%

DV × firm_chars:
_merge
both          391970
left_only      11430
right_only         0
Name: count, dtype: int64
  firm_chars missing: 2.8%


## Step 11　导出 Stata 文件

In [41]:
# invoice_panel
inv_cols = ['firm_id', 'product_id', 'city', 'year',
            'value', 'qty', 'p_buy', 'ln_p_buy', 'p_net', 'ln_p_net', 'ln_qty', 'n_rows']
dv[inv_cols].to_stata(BASE / 'invoice_panel.dta', write_index=False, version=118)
print('invoice_panel.dta:', len(dv))

# market_conds
mkt_cols = ['product_id', 'city', 'year',
            'mkt_value', 'mkt_qty', 'p_mkt', 'ln_p_mkt', 'ln_mkt_qty',
            'n_buyers', 'ln_n_buyers',
            'sell_value', 'sell_qty', 'n_sellers', 'ln_n_sellers']
market_conds[mkt_cols].to_stata(BASE / 'market_conds.dta', write_index=False, version=118)
print('market_conds.dta: ', len(market_conds))

# firm_chars
fc_cols = ['firm_id', 'year', 'firm_total_output', 'firm_total_outsource',
           'n_products', 'is_intermediary', 'ln_firm_output', 'ln_firm_outsource']
firm_chars[fc_cols].to_stata(BASE / 'firm_chars.dta', write_index=False, version=118)
print('firm_chars.dta:   ', len(firm_chars))

print('\n=== 清洗完成 ===')

invoice_panel.dta: 403400
market_conds.dta:  1066579
firm_chars.dta:    12339537

=== 清洗完成 ===


清理完数据后，就要进入02_reg
